# Demo 05: input-gradient spectra of an encoding axis

The input gradient $\partial \hat y / \partial \text{image}$ of a site's encoding model says which pixels the model's
prediction is sensitive to. Its radial Fourier power profile summarizes *at which spatial frequencies* that
sensitivity lives; the paper compares models by how concentrated the profile is (Fig. 5, "spectral flatness" and the
participation ratio). The upstream generator is `scripts/gradients/encoding_model_gradmap_freq_analysis.py`
(10 seed images per readout, GPU); its per-(site, model) profiles are cached in
`preprocessed_data/gradient_freq.pkl` (`L.load_gradient_freq()`) and the 2-D gradient maps of three example sites
in `preprocessed_data/grad_maps/` (`L.load_grad_maps(monkey, unit)`).

This notebook recomputes the gradients for 3 of the 10 seed images under ResNet-50 and robust ResNet-50, derives
the profiles with the vendored recipe in `pnc.preproc.fft_utils`, and checks both against the caches. There is no
reduced scale to speak of (a gradient is one backward pass); `FULL_SCALE = True` simply uses all 10 seeds.

In [1]:
import sys
sys.path[:0] = ['..', 'notebooks']        # notebooks/nbsetup.py holds the setup shared by every notebook
from nbsetup import setup

OUT_DIR = setup(needs='source')          # single-thread BLAS, Agg backend, repo on the path, data check
import os, sys, time
from pathlib import Path
from pnc import paths
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from pnc.utils import apply_figure_style
apply_figure_style(); plt.rcParams['figure.dpi'] = 90
torch.set_num_threads(max(1, os.cpu_count() // 2)); torch.manual_seed(0); np.random.seed(0)
print('source_data       =', paths.source_data())
print('preprocessed_data =', paths.preprocessed_data())
from IPython.display import display

source_data       = /private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/source_data
preprocessed_data = /private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/preproc_build


In [2]:
FULL_SCALE = False
MONKEY, UNIT = 'red', 9
MODELS = ['resnet50', 'resnet50_robust']
SEED_IDX = list(range(10)) if FULL_SCALE else [0, 1, 3]
import _demo_utils as U
from pnc.utils import MODEL_SHORT_NAMES, get_model_color
from pnc.preproc import loader as L
from pnc.preproc.fft_utils import gradient_radial_profile, spectral_flatness
from figures.main.fig5_predictors import spectral_pr, FMIN, FMAX       # participation ratio + the 1-111 cyc/img band
import inspect
print(inspect.getsource(spectral_pr)); print(inspect.getsource(spectral_flatness))
print('band:', FMIN, FMAX)

def spectral_pr(p):                                     # participation ratio: effective number of frequency bands
    P = np.asarray(p, float)[FMIN:FMAX]                 # (= n / (1 + CoV^2), an exact transform of spectral CoV)
    return float(P.sum() ** 2 / (P ** 2).sum())

def spectral_flatness(profile, fmin=1, fmax=112):
    """Wiener spectral flatness (geometric/arithmetic mean) of a radial power profile
    over the [fmin, fmax) frequency-bin band. 1 = white/flat, low = concentrated."""
    p = np.clip(np.asarray(profile, float)[fmin:fmax], 1e-30, None)
    return float(np.exp(np.mean(np.log(p))) / p.mean())

band: 1 112


## 1. Gradients

The upstream script feeds the seed images through the model's preprocessing (`ToTensor -> Resize(224) -> Normalize`)
and differentiates the target-site prediction with respect to that *normalized* tensor; the same is done here
(`EncodingObjective.from_normalized`), so the maps are directly comparable with the cache.

In [3]:
from PIL import Image
seed_paths = [U.stimulus_path(U.SEED_NAMES[i]) for i in SEED_IDX]
grads, preds = {}, {}
for m in MODELS:
    backbone, transform = U.load_backbone(m)
    obj = U.EncodingObjective(m, MONKEY, UNIT, backbone=backbone)
    xn = torch.stack([transform(Image.open(p).convert('RGB')) for p in seed_paths]).requires_grad_(True)
    pred = obj.from_normalized(xn)
    pred.sum().backward()                                # images are independent, so one backward of the sum gives every per-image gradient
    grads[m] = xn.grad.detach().numpy()                   # (n_seeds, 3, 224, 224)
    preds[m] = pred.detach().numpy()
    obj.close()
    print(f'{m:16s} layer {obj.layer:22s} predictions {preds[m].round(3)}  |grad| L2 per image {np.linalg.norm(grads[m].reshape(len(SEED_IDX), -1), axis=1).round(3)}')

/private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FeatureFetcher hooks all freed
resnet50         layer .layer4.Bottleneck0    predictions [ 0.52   0.763 -0.268]  |grad| L2 per image [0.91  0.585 0.452]


/private/tmp/claude-501/-Users-jacobprince-KonkLab-Dropbox-Jacob-Prince-Research-Prince-Accentuate-VVS-Manuscript/bca6882d-ec2d-4dbd-beb1-0d67d8b1fcc7/scratchpad/env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


FeatureFetcher hooks all freed
resnet50_robust  layer .layer4.Bottleneck2    predictions [ 1.147  0.743 -0.166]  |grad| L2 per image [0.043 0.028 0.044]


## 2. Check against the cached gradient maps

`L.load_grad_maps('red', 9)[model]` is `(10, 3, 224, 224)` from the cluster run; pixel-wise correlation with the
maps just computed for the same seeds.

In [4]:
gm = L.load_grad_maps(MONKEY, UNIT)
for m in MODELS:
    rs = [np.corrcoef(grads[m][j].ravel(), gm[m][i].ravel())[0, 1] for j, i in enumerate(SEED_IDX)]   # cache holds all 10 seeds: j this run, i the seed
    print(f'{m:16s} correlation with cached gradient maps per seed: {np.round(rs, 4)}')

resnet50         correlation with cached gradient maps per seed: [0.9931 0.995  0.9974]
resnet50_robust  correlation with cached gradient maps per seed: [0.9988 0.9996 0.9997]


## 3. Radial Fourier power profiles, flatness and participation ratio

`gradient_radial_profile` (vendored verbatim from the upstream toolkit) averages the gradient over color channels,
takes $|\mathrm{FFT}|^2$ and averages it in integer-radius bins about the image centre. Over the 1-111 cyc/img band,
*flatness* is the geometric / arithmetic mean of the profile and the *participation ratio*
$\mathrm{PR} = (\sum_f P_f)^2 / \sum_f P_f^2$ is the effective number of frequency bands carrying the power
(both from `figures/main/fig5_predictors.py`).

In [5]:
GF = L.load_gradient_freq()
freqs = GF['freqs']
cached = {m: next(g for g in GF['gradients'] if g['monkey'] == MONKEY and g['unit'] == UNIT and g['model'] == m) for m in MODELS}   # per (site, model): mean profile over 10 seeds
profiles = {m: np.stack([gradient_radial_profile(g)[0] for g in grads[m]]) for m in MODELS}   # returns (profile, freqs); keep the profile
rows = []
for m in MODELS:
    for j, i in enumerate(SEED_IDX):
        rows.append(dict(model=m, seed=i, flatness=spectral_flatness(profiles[m][j]), PR=spectral_pr(profiles[m][j])))
    rows.append(dict(model=m, seed='cached mean profile (10 seeds)', flatness=spectral_flatness(cached[m]['profile_mean']), PR=spectral_pr(cached[m]['profile_mean'])))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print('\nprofile of the cache for these seeds vs recomputed (max relative deviation):')
gm_prof = {m: np.stack([gradient_radial_profile(gm[m][i])[0] for i in SEED_IDX]) for m in MODELS}
for m in MODELS:
    rel = np.abs(profiles[m] - gm_prof[m])[:, FMIN:FMAX] / gm_prof[m][:, FMIN:FMAX]   # compare inside the analysis band only
    print(f'  {m:16s} {rel.max():.3g}')

          model                           seed  flatness     PR
       resnet50                              0     0.848 91.737
       resnet50                              1     0.871 96.310
       resnet50                              3     0.883 98.360
       resnet50 cached mean profile (10 seeds)     0.871 95.728
resnet50_robust                              0     0.690 45.223
resnet50_robust                              1     0.550 32.449
resnet50_robust                              3     0.564 35.533
resnet50_robust cached mean profile (10 seeds)     0.638 44.550

profile of the cache for these seeds vs recomputed (max relative deviation):
  resnet50         0.0707
  resnet50_robust  0.0336


In [6]:
def norm_profile(p):                                     # normalize to unit power over the band (figures/main/fig5_grad.normalize_row)
    p = np.asarray(p, float); return p / p[FMIN:FMAX].sum()

fig = plt.figure(figsize=(13, 6.2))
gs = fig.add_gridspec(2, len(SEED_IDX) + 2, height_ratios=[1, 1.1])
for r, m in enumerate(MODELS):
    for j, i in enumerate(SEED_IDX):
        ax = fig.add_subplot(gs[r, j])
        sal = np.sqrt((grads[m][j] ** 2).sum(0)); ax.imshow(sal / np.percentile(sal, 99), cmap='inferno', vmin=0, vmax=1)   # gradient magnitude over color channels
        ax.set_axis_off(); ax.set_title(f'{MODEL_SHORT_NAMES[m]}, seed {i}\npred {preds[m][j]:+.2f}', fontsize=7)
ax = fig.add_subplot(gs[:, len(SEED_IDX):])
x = freqs[FMIN:FMAX]
ax.plot(x, norm_profile(GF['natural_profile'])[FMIN:FMAX], color='k', lw=1, ls=':', label='natural images (cached)')
for m in MODELS:
    c = get_model_color(m)
    for j in range(len(SEED_IDX)):
        ax.plot(x, norm_profile(profiles[m][j])[FMIN:FMAX], color=c, lw=0.6, alpha=0.5)
    ax.plot(x, norm_profile(cached[m]['profile_mean'])[FMIN:FMAX], color=c, lw=2, ls='--',
            label=f'{MODEL_SHORT_NAMES[m]}: cached mean (PR {spectral_pr(cached[m]["profile_mean"]):.1f}, flat {spectral_flatness(cached[m]["profile_mean"]):.2f})')
    ax.plot([], [], color=c, lw=0.6, label=f'{MODEL_SHORT_NAMES[m]}: this run, per seed')   # legend entry for the thin per-seed lines
ax.set(xscale='log', yscale='log', xlabel='spatial frequency (cycles / image)', ylabel='normalized radial power',
       title=f'input-gradient spectra, Monkey R site {UNIT}')
ax.legend(fontsize=6); plt.tight_layout(); display(plt.gcf()); plt.close()

<Figure size 1300x620 with 7 Axes>

The robust model's gradient is concentrated at low spatial frequencies (a steeper spectrum, smaller participation
ratio), while the standard ResNet-50 spreads its sensitivity into high frequencies -- the property the paper relates
to adversarial reach (demo 04) and to how well a model's accentuations control the neurons.